In [1]:
%%capture
!pip install unidecode

In [2]:
import torch
from google.colab import drive

drive.mount("/drive")

Mounted at /drive


In [3]:
from unidecode import unidecode

In [4]:
!ls  /drive/MyDrive/names

Arabic.txt   English.txt  Irish.txt	Polish.txt	Spanish.txt
Chinese.txt  French.txt   Italian.txt	Portuguese.txt	Vietnamese.txt
Czech.txt    German.txt   Japanese.txt	Russian.txt
Dutch.txt    Greek.txt	  Korean.txt	Scottish.txt


In [5]:
!cat /drive/MyDrive/names/Arabic.txt

Khoury
Nahas
Daher
Gerges
Nazari
Maalouf
Gerges
Naifeh
Guirguis
Baba
Sabbagh
Attia
Tahan
Haddad
Aswad
Najjar
Dagher
Maloof
Isa
Asghar
Nader
Gaber
Abboud
Maalouf
Zogby
Srour
Bahar
Mustafa
Hanania
Daher
Tuma
Nahas
Saliba
Shamoon
Handal
Baba
Amari
Bahar
Atiyeh
Said
Khouri
Tahan
Baba
Mustafa
Guirguis
Sleiman
Seif
Dagher
Bahar
Gaber
Harb
Seif
Asker
Nader
Antar
Awad
Srour
Shadid
Hajjar
Hanania
Kalb
Shadid
Bazzi
Mustafa
Masih
Ghanem
Haddad
Isa
Antoun
Sarraf
Sleiman
Dagher
Najjar
Malouf
Nahas
Naser
Saliba
Shamon
Malouf
Kalb
Daher
Maalouf
Wasem
Kanaan
Naifeh
Boutros
Moghadam
Masih
Sleiman
Aswad
Cham
Assaf
Quraishi
Shalhoub
Sabbag
Mifsud
Gaber
Shammas
Tannous
Sleiman
Bazzi
Quraishi
Rahal
Cham
Ghanem
Ghanem
Naser
Baba
Shamon
Almasi
Basara
Quraishi
Bata
Wasem
Shamoun
Deeb
Touma
Asfour
Deeb
Hadad
Naifeh
Touma
Bazzi
Shamoun
Nahas
Haddad
Arian
Kouri
Deeb
Toma
Halabi
Nazari
Saliba
Fakhoury
Hadad
Baba
Mansour
Sayegh
Antar
Deeb
Morcos
Shalhoub
Sarraf
Amari
Wasem
Ganim
Tuma
Fakhoury
Hadad
Hakimi
Nader
Sa

In [6]:
import os
from glob import glob

In [7]:
root_dir = "/drive/MyDrive/names"
file_names = glob("*.txt", root_dir=root_dir)
unique_labels = sorted([os.path.splitext(file_name)[0] for file_name in file_names])
n_labels = len(unique_labels)

idx2label = {idx:label for idx, label in enumerate(unique_labels)}
label2idx = {label:idx  for idx, label in idx2label.items()}


In [8]:
idx2label

{0: 'Arabic',
 1: 'Chinese',
 2: 'Czech',
 3: 'Dutch',
 4: 'English',
 5: 'French',
 6: 'German',
 7: 'Greek',
 8: 'Irish',
 9: 'Italian',
 10: 'Japanese',
 11: 'Korean',
 12: 'Polish',
 13: 'Portuguese',
 14: 'Russian',
 15: 'Scottish',
 16: 'Spanish',
 17: 'Vietnamese'}

In [9]:
def replace(name, chars, target):
  for char in chars:
    name = name.replace(char, target)
  return name

In [10]:
X_names = []
Y_labels = []

for file_name in file_names:
  with open(os.path.join(root_dir, file_name), "rt", encoding='utf-8') as f:
    for line in f:
      name = line.strip().lower()
      name = unidecode(name)

      if name == 'to the first page':
        continue

      name = replace(name, [",", "1", "/b", ":", "\xa0"], '')
      name = replace(name, ['-'], '')

      X_names.append(name)
      Y_labels.append(os.path.splitext(file_name)[0])

In [11]:
pad_token = '.'
pad_token_id = 0

unique_chars = [pad_token] + sorted(set(''.join(X_names)))
idx2char = {idx:char for idx, char in enumerate(unique_chars)}
char2idx = {char:idx for idx, char in idx2char.items()}

def encode(name: str) -> list[int]:
  return [char2idx[char] for char in name]

def decode(ids: list[int]) -> str:
  return ''.join(idx2char[i] for i in ids)


In [12]:
len(unique_chars)

30

In [13]:
Y = [label2idx[(label)] for label in Y_labels]
X = [encode(name) for name in X_names]

In [14]:
for x, x_name, y, y_label in zip(X[:5], X_names[:5], Y[:5], Y_labels[:5]):
  print(f"{str(x):<50} -> {x_name:30} \t\t{y} -> {y_label}")

[4, 5, 4, 5, 14, 18]                               -> ababko                         		14 -> Russian
[4, 5, 4, 8, 25]                                   -> abaev                          		14 -> Russian
[4, 5, 4, 10, 28, 4, 17]                           -> abagyan                        		14 -> Russian
[4, 5, 4, 12, 7, 24, 15, 12, 17]                   -> abaidulin                      		14 -> Russian
[4, 5, 4, 12, 7, 24, 15, 15, 12, 17]               -> abaidullin                     		14 -> Russian


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X_tr, X_ts, Y_tr, Y_ts = train_test_split(X, Y, test_size=0.2, stratify=Y)

In [17]:
from torch.utils.data import Dataset, DataLoader

class NamesDataset(Dataset):
  def __init__(self, X, Y):
    self.X = X
    self.Y = Y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.Y[idx]


Dtr = NamesDataset(X_tr, Y_tr)
Dts = NamesDataset(X_ts, Y_ts)



In [18]:
len(Dtr)

16040

In [19]:
len(Dts)

4011

In [20]:
Dtr[0]

([7, 4, 16, 11, 4, 12, 17], 8)

In [21]:
global_max_n = 20

def collate_fn(batch):
  X_batch, Y_batch = zip(*batch)
  max_len = max(len(x) for x in X_batch)
  max_len = global_max_n if max_len > global_max_n else max_len

  padded_X = [x + [pad_token_id] * (max_len - len(x)) for x in X_batch]

  return torch.tensor(padded_X), torch.tensor(Y_batch)

Dltr = DataLoader(Dtr, batch_size=4, shuffle=True, drop_last=True, collate_fn=collate_fn)
Dlts = DataLoader(Dts, batch_size=4, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [22]:
for batch in Dltr:
  print(batch)
  break

(tensor([[ 5,  8, 15, 18, 21, 18, 22, 22, 18, 25],
        [22,  4, 15, 18, 16, 18, 17,  0,  0,  0],
        [23, 24, 16,  4,  0,  0,  0,  0,  0,  0],
        [ 5,  8, 15, 18, 22,  8, 15, 22, 14, 28]]), tensor([14,  6,  0, 14]))


In [36]:
# Define the model
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class NamesClassifier(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.emb = nn.Embedding(self.config.vocab_size, self.config.n_embd)
    self.conv = nn.Conv1d(self.config.n_embd, self.config.n_conv_channels, self.config.kernel_size)
    self.max_pool = nn.AdaptiveMaxPool1d(1)
    self.drop = nn.Dropout(self.config.drop_rate)
    self.fc = nn.Linear(self.config.n_conv_channels, self.config.n_labels)

  def forward(self, x):
    x = self.emb(x)
    x = x.permute(0, 2, 1)
    x = self.conv(x)
    x = self.max_pool(x)
    x = self.drop(x.squeeze(-1)) # Changed .squeeze() to .squeeze(-1)
    x = self.fc(x)
    return x


@dataclass
class Config:
  vocab_size: int
  n_embd: int
  n_conv_channels: int
  kernel_size: int
  drop_rate: float
  n_labels: int


config = Config(vocab_size=30, n_embd=16, n_conv_channels=32, kernel_size=3, drop_rate=0.5, n_labels=18)
model = NamesClassifier(config)

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [46]:
import torch.optim as optim
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score
import numpy as np

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss()

n_epochs = 20

model.train()
for epoch in range(1, n_epochs + 1):
  total_loss = 0.0
  for x_batch, y_batch in Dltr:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    optimizer.zero_grad()

    logits = model(x_batch)
    loss = criterion(logits, y_batch)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  avg_loss = total_loss / len(Dltr)
  print(f"Epoch {epoch}/{n_epochs} - Train loss: {avg_loss:.4f}")


Epoch 1/20 - Train loss: 1.0014
Epoch 2/20 - Train loss: 0.9941
Epoch 3/20 - Train loss: 0.9970
Epoch 4/20 - Train loss: 0.9833
Epoch 5/20 - Train loss: 0.9762
Epoch 6/20 - Train loss: 0.9735
Epoch 7/20 - Train loss: 0.9662
Epoch 8/20 - Train loss: 0.9614
Epoch 9/20 - Train loss: 0.9641
Epoch 10/20 - Train loss: 0.9539
Epoch 11/20 - Train loss: 0.9530
Epoch 12/20 - Train loss: 0.9576
Epoch 13/20 - Train loss: 0.9540
Epoch 14/20 - Train loss: 0.9523
Epoch 15/20 - Train loss: 0.9530
Epoch 16/20 - Train loss: 0.9471
Epoch 17/20 - Train loss: 0.9475
Epoch 18/20 - Train loss: 0.9438
Epoch 19/20 - Train loss: 0.9426
Epoch 20/20 - Train loss: 0.9383


In [47]:
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
  for x_batch, y_batch in Dlts:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    logits = model(x_batch)
    preds = torch.argmax(logits, dim=1)

    all_preds.extend(preds.cpu().numpy())
    all_true.extend(y_batch.cpu().numpy())

accuracy = accuracy_score(all_true, all_preds)
print(f"Test Accuracy: {accuracy:.4f} -> {accuracy*100:.2f}%")

Test Accuracy: 0.7537 -> 75.37%


In [48]:
def predict_name(name: str):
  name = name.strip().lower()
  name = replace(name, [",", "1", "/b", ":", "\xa0"], '')
  name = replace(name, ['-'], '')
  name = unidecode(name)

  encoded = encode(name)

  if len(encoded) > global_max_n:
    encoded = encoded[:global_max_n]
  else:
    encoded += [pad_token_id] * (global_max_n - len(encoded))

  x = torch.tensor([encoded]).to(device)

  model.eval()
  with torch.no_grad():
    logits = model(x)
    probs = F.softmax(logits, dim=1)
    pred_idx = torch.argmax(probs, dim=1).item()

  predicted_label = idx2label[pred_idx]
  proba = probs[0][pred_idx].item()

  top3_idx = torch.topk(probs, 3).indices[0].tolist()
  print(f"Name: {name.capitalize()}")
  for i, idx in enumerate(top3_idx, 1):
    label = idx2label[idx]
    prob = probs[0][idx].item()
    print(f"  {i}. {label}: {prob*100:.1f}%")
  print()

In [49]:
test_names = ["Sanjar", "Mei", "Dubois", "Müller", "Takahashi", "Patel", "Kowalski"]

for name in test_names:
    predict_name(name)

Name: Sanjar
  1. Russian: 44.4%
  2. Arabic: 24.1%
  3. English: 16.7%

Name: Mei
  1. Chinese: 52.3%
  2. Japanese: 13.3%
  3. Korean: 12.4%

Name: Dubois
  1. Russian: 48.0%
  2. English: 12.3%
  3. Greek: 11.1%

Name: Muller
  1. English: 65.5%
  2. Russian: 12.4%
  3. German: 10.1%

Name: Takahashi
  1. Japanese: 91.0%
  2. Russian: 7.2%
  3. Arabic: 0.9%

Name: Patel
  1. English: 37.4%
  2. Russian: 32.4%
  3. Czech: 11.5%

Name: Kowalski
  1. Czech: 40.6%
  2. Russian: 37.7%
  3. Polish: 16.6%

